# Lab 1 — Problem Framing và vai trò dữ liệu

   **Week 1 · End-to-End Machine Learning qua câu chuyện Titanic**
   Trường Đại học Sài Gòn — Khoa Công nghệ Thông tin

   | | |
   |---|---|
   | Episode tương ứng | 1, 2 |
   | Chuẩn đầu ra | W1-CLO1, W1-CLO7 |
   | Thời lượng | 45 phút |
   | Tổng điểm | 8 |
   | Bản này | Bản làm bài của sinh viên |

   Episode 1 và 2 đặt ra một thứ tự không được đảo: **Problem Framing → Data Roles → Model**.
Lab này biến thứ tự đó thành code. Sinh viên chưa huấn luyện gì cả — chỉ tách target khỏi
feature, loại identifier, và loại mọi biến không vượt qua Time-of-Prediction Test.

Ba hàm dưới đây chính là hai bước đầu trong *Pseudocode chuẩn* ở Episode 10.

   ---

   ## Quy tắc làm bài

   1. Chỉ sửa các ô được đánh dấu `# === BÀI LÀM ===`. Không đổi tên hàm và không đổi thứ tự tham số — bộ chấm gọi đúng những tên đó.
   2. Mỗi hàm phải **thuần**: chỉ dùng tham số truyền vào, không đọc biến toàn cục, không đọc file bên trong hàm.
   3. Chạy ô tự kiểm tra ngay dưới mỗi bài. Ô đó chạy trên dữ liệu nhỏ viết sẵn nên không cần `train.csv`.
   4. Nộp đúng file notebook này, đổi tên thành `<MSSV>_lab01.ipynb`.

## Thư viện

In [ ]:
import os
import sys

# Tự tìm thư mục gốc của bộ lab để notebook chạy được dù mở từ đâu
# (thư mục labs/, labs/solutions/, hay Colab sau khi giải nén).
GOC = os.path.abspath(os.getcwd())
while GOC != os.path.dirname(GOC) and not os.path.exists(os.path.join(GOC, "w1lab.py")):
    GOC = os.path.dirname(GOC)
if os.path.exists(os.path.join(GOC, "w1lab.py")):
    sys.path.insert(0, GOC)
else:
    print("Cảnh báo: không tìm thấy w1lab.py. Hãy mở notebook từ trong thư mục bộ lab.")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
pd.set_option("display.width", 120)

## Nạp dữ liệu

Bài lab dùng `train.csv` của Kaggle Titanic. Ô này chỉ phục vụ phần khám phá;
bộ chấm **không** chạy ô này, nên nếu máy chưa có dữ liệu thì các ô bài làm vẫn chấm được.

In [ ]:
# Đặt `train.csv` tải từ Kaggle vào thư mục `data/` ở gốc bộ lab.
DATA_PATH = os.path.join(GOC, "data", "train.csv")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Chưa có {DATA_PATH}.\n"
        "Tải train.csv từ https://www.kaggle.com/competitions/titanic/data "
        "rồi đặt vào thư mục data/ ở gốc bộ lab.")

df = pd.read_csv(DATA_PATH)
print(f"{len(df)} bản ghi, {df.shape[1]} cột")
df.head()

---

     ## Bài 1. audit_feature_roles — 2.5 điểm

     Viết hàm phân loại vai trò của từng cột trong một DataFrame theo bốn nhãn:

- `"target"` — đại lượng cần dự đoán;
- `"identifier"` — chỉ định danh bản ghi;
- `"leakage"` — thông tin **không tồn tại** tại thời điểm dự đoán;
- `"feature"` — mọi cột còn lại.

Quy ước cho bài lab: target là `Survived`, identifier là `PassengerId`, và mọi cột
có tên nằm trong `posthoc_columns` là leakage. Hàm phải hoạt động cả khi
DataFrame thiếu vài cột hoặc có thêm cột lạ.

In [ ]:
# === BÀI LÀM ===
def audit_feature_roles(df, target="Survived", identifier="PassengerId",
                        posthoc_columns=("RescueBoatNumber", "SurvivalCertificateIssued")):
    """Trả về dict {tên_cột: vai_trò} cho mọi cột của df."""

    # TODO: duyệt df.columns và gán đúng một trong bốn nhãn cho mỗi cột.
    # Gợi ý: kiểm tra target trước, rồi identifier, rồi posthoc_columns, còn lại là feature.
    raise NotImplementedError("Sinh viên cài đặt")

**Tự kiểm tra.** Ô dưới phải chạy không báo lỗi.

In [ ]:

mini = pd.DataFrame({"PassengerId": [1], "Survived": [0], "Age": [22.0],
                     "RescueBoatNumber": [7]})
vai_tro = audit_feature_roles(mini)
assert vai_tro["Survived"] == "target"
assert vai_tro["PassengerId"] == "identifier"
assert vai_tro["RescueBoatNumber"] == "leakage"
assert vai_tro["Age"] == "feature"
print("Đạt.")

---

     ## Bài 2. define_problem — 2.5 điểm

     Viết hàm tách bài toán thành `X` và `y`, dùng lại kết quả của `audit_feature_roles`.

`X` chỉ được chứa các cột có vai trò `"feature"` — **không** target, **không** identifier,
**không** biến hậu nghiệm. Thứ tự cột trong `X` giữ nguyên như trong `df`.
Nếu DataFrame không có cột target thì ném `KeyError`.

Đây chính là bước 2 của *Pseudocode chuẩn*: `X, y = define_problem(df)`.

In [ ]:
# === BÀI LÀM ===
def define_problem(df, target="Survived", identifier="PassengerId",
                   posthoc_columns=("RescueBoatNumber", "SurvivalCertificateIssued")):
    """Trả về (X, y): DataFrame feature hợp lệ và Series target."""

    # TODO: gọi audit_feature_roles, lấy danh sách cột có vai trò "feature",
    # trả về (df[cot_feature], df[target]). Nhớ ném KeyError nếu thiếu target.
    raise NotImplementedError("Sinh viên cài đặt")

**Tự kiểm tra.** Ô dưới phải chạy không báo lỗi.

In [ ]:

mini = pd.DataFrame({"PassengerId": [1, 2], "Survived": [0, 1],
                     "Age": [22.0, 38.0], "Sex": ["male", "female"]})
X, y = define_problem(mini)
assert list(X.columns) == ["Age", "Sex"], f"X sai cột: {list(X.columns)}"
assert y.tolist() == [0, 1]
assert "Survived" not in X.columns and "PassengerId" not in X.columns
print("Đạt.")

---

     ## Bài 3. add_derived_features — 3 điểm

     Episode 2 chỉ ra ba biến dẫn xuất có nghĩa. Viết hàm thêm chúng vào một **bản sao**
của DataFrame (không sửa `df` gốc):

- `FamilySize = SibSp + Parch + 1`
- `HasCabin = 1` nếu `Cabin` được ghi nhận, `0` nếu thiếu
- `Title` — danh xưng trích từ `Name`, tức chuỗi nằm giữa dấu phẩy và dấu chấm đầu tiên,
  đã cắt khoảng trắng. Ví dụ `"Braund, Mr. Owen Harris"` → `"Mr"`.

Nếu một cột nguồn không tồn tại thì bỏ qua biến dẫn xuất tương ứng, không ném lỗi.

In [ ]:
# === BÀI LÀM ===
def add_derived_features(df):
    """Trả về bản sao của df có thêm FamilySize, HasCabin, Title."""

    # TODO: out = df.copy() rồi thêm từng cột nếu cột nguồn có mặt.
    # Gợi ý cho Title: dùng .str.extract với biểu thức chính quy, hoặc .str.split.
    raise NotImplementedError("Sinh viên cài đặt")

**Tự kiểm tra.** Ô dưới phải chạy không báo lỗi.

In [ ]:

mini = pd.DataFrame({
    "Name": ["Braund, Mr. Owen Harris", "Palsson, Master. Gosta Leonard"],
    "SibSp": [1, 3], "Parch": [0, 1], "Cabin": [None, "C85"]})
out = add_derived_features(mini)
assert out["FamilySize"].tolist() == [2, 5]
assert out["HasCabin"].tolist() == [0, 1]
assert out["Title"].tolist() == ["Mr", "Master"]
assert "FamilySize" not in mini.columns, "Hàm đã sửa df gốc — phải làm trên bản sao"
print("Đạt.")

---

## Khám phá trên dữ liệu thật

Chạy trên `train.csv` thật và đối chiếu với data dictionary ở Episode 2.

In [ ]:
X, y = define_problem(df)
print("Số feature hợp lệ:", X.shape[1])
print("Tỷ lệ lớp dương:", round(y.mean(), 4))

df_mo_rong = add_derived_features(df)
print("\nPhân bố Title:")
print(df_mo_rong["Title"].value_counts().head(8))
print("\nTỷ lệ có Cabin:", round(df_mo_rong["HasCabin"].mean(), 4))

---

## Chốt bài

Kết thúc lab này, sinh viên đã có `X` và `y` **sạch về vai trò dữ liệu**: không target,
 không identifier, không biến hậu nghiệm. Đó là điều kiện cần trước khi được phép nghĩ
 tới model. Lab 2 sẽ dùng chính `X`, `y` này để dựng mức tham chiếu.